# Agente conversacional simple con LangChain


## ¿Qué es un Agente?

En el contexto de la Inteligencia Artificial y los Modelos de Lenguaje Grandes (LLM), un **agente** es un sistema que puede **razonar** sobre una tarea y **decidir** qué **herramientas** usar para resolverla. A diferencia de un LLM que solo genera texto, un agente tiene la capacidad de:

1.  **Entender** una pregunta o instrucción.
2.  **Planificar** una serie de pasos para lograr el objetivo.
3.  **Ejecutar** acciones utilizando herramientas (como calculadoras, búsquedas en internet, bases de datos, etc.).
4.  **Observar** los resultados de esas acciones.
5.  **Corregir** o ajustar su plan si es necesario.

## ¿Qué es LangChain?

**LangChain** es un _framework_ (un conjunto de herramientas, componentes y patrones) que facilita el desarrollo de aplicaciones impulsadas por modelos de lenguaje grandes (LLMs).

Su objetivo principal es ayudar a los desarrolladores a:

*   **Conectar** LLMs con fuentes de datos externas (APIs, bases de datos).
*   **Permitir** a los LLMs interactuar con su entorno.
*   **Construir** **agentes** que puedan razonar y tomar decisiones, usando los LLMs como su "cerebro" y herramientas para interactuar con el mundo.

### Instalación de Librerías

Primero, instalamos las librerías necesarias para trabajar con LangChain y el modelo de OpenAI a través de OpenRouter. `langchain_openai` proporciona integraciones con modelos de OpenAI, y `langchain-openrouter` es para la integración específica con OpenRouter.

In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 20.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [ ]:
!pip install -U langchain langchain-openrouter

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.8/161.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.3/837.3 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.4 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.5
    Uninstalling pydantic_core-2.46.5:
      Successfully uninstalled pydantic_core-2.46.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.5
    Uninstalling pydantic-2.13.5:
      Successfully uninstalled pydantic-2.13.5
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.18
    Uninstalling langchain-1.3.18:
      Successfully uninstalled langchain-1.3.18


### Importación de Módulos

Aquí importamos los módulos clave que utilizaremos:

*   `create_agent` de `langchain.agents`: Para construir nuestro agente.
*   `tool` de `langchain.tools`: Un decorador para definir funciones que el agente puede usar como herramientas.
*   `ChatOpenAI` de `langchain_openai`: La clase que usaremos para interactuar con los modelos de chat compatibles con la API de OpenAI, incluyendo los de OpenRouter.
*   `userdata` de `google.colab`: Para acceder de forma segura a las claves de API almacenadas en Colab.
*   `os`: Para interactuar con variables de entorno.
*   `requests`: Una librería HTTP para hacer solicitudes web (aunque no se usa directamente en el agente, es común en este tipo de desarrollo).
*   `OpenAI` de `openai`: El cliente oficial de OpenAI, que usaremos para configurar la conexión con OpenRouter.

In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from google.colab import userdata
import os
import requests
from openai import OpenAI

### Configuración de la Clave de API

Obtenemos la clave de API de OpenRouter de los secretos de Colab y la configuramos como una variable de entorno. Esto es crucial para autenticarse con el servicio de OpenRouter.

In [ ]:
os.environ["OPENROUTER_KEY"] = userdata.get('OPENROUTER_KEY')

### Inicialización del Cliente OpenRouter

Creamos una instancia del cliente `OpenAI` apuntando a la URL base de OpenRouter y usando nuestra clave de API. Esto nos permite hacer llamadas a los modelos disponibles en OpenRouter como si fueran modelos de OpenAI.

In [ ]:
# Configurar el cliente para OpenRouter
# Inicializar el cliente leyendo desde la variable de entorno
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.environ.get("OPENROUTER_KEY")
)

### Definición de Herramienta (`calcular`)

Definimos una función simple llamada `calcular` que toma una expresión matemática como cadena y la evalúa. El decorador `@tool` de LangChain la convierte en una herramienta que nuestro agente puede utilizar. La docstring de la función (`"""Calcula una expresión matemática simple."""`) es importante porque LangChain la usa para que el LLM sepa cuándo y cómo usar la herramienta.

In [ ]:
# 1. Definimos una herramienta
@tool
def calcular(expresion: str) -> str:
    """Calcula una expresión matemática simple."""
    return str(eval(expresion))

### Inicialización del Modelo de Lenguaje

Aquí instanciamos el modelo de lenguaje que el agente utilizará para su razonamiento. Estamos usando `ChatOpenAI` de `langchain_openai`, especificando el modelo `cohere/north-mini-code:free` (disponible en OpenRouter), nuestra `api_key` y la `base_url` de OpenRouter.

In [ ]:
# 3. Creamos el agente
agent = create_agent(
    model=model,
    tools=[calcular],
    system_prompt="""
    Sos un asistente.
    Si la pregunta requiere un cálculo matemático,
    utilizá la herramienta calcular.
    Si no, respondé directamente.
    """
)

### Ejecución del Agente

Ahora, invocamos el agente con una pregunta de usuario. El agente utilizará su lógica (definida por el modelo y el `system_prompt`) para decidir si necesita usar la herramienta `calcular` o si puede responder directamente. En este caso, al ser un cálculo, debería usar la herramienta.

El resultado final es el contenido del último mensaje generado por el agente.

In [ ]:
model = ChatOpenAI(
    model="cohere/north-mini-code:free",
    api_key=os.environ.get("OPENROUTER_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

### Creación del Agente

Finalmente, creamos el agente utilizando `create_agent`.

*   Le pasamos el `model` que definimos (el cerebro del agente).
*   Le proporcionamos la lista de `tools` disponibles (en este caso, solo `calcular`).
*   Le damos un `system_prompt` que guía su comportamiento, indicándole que use la herramienta `calcular` para preguntas matemáticas y que responda directamente si no lo son.

In [ ]:
# 4. Ejecutamos el agente
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "¿Cuánto es 123 * 47?"
        }
    ]
})

print(result["messages"][-1].content)

El resultado de \(123 \times 47\) es **5781**.
